In [ ]:
import os
import copy
import subprocess
from glob import glob
from itertools import product
from datetime import datetime
from pathlib import Path
from string import Template
from utils.notebook import isnotebook
if isnotebook():
    home_dir = os.path.expanduser("~")
    os.chdir(os.path.join(home_dir, "aiwq"))

    # Autoreload modified packages
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

# Inline plotting setup
%matplotlib inline
%config InlineBackend.figure_formats = ['pdf', 'svg']
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Markdown, display
from AI_WQ_package import forecast_submission
from src.utils.data_io import *
from src.viz.viz_utils_pbc import *
from models.utils.general_util import printf
from models.utils.eval_util import get_target_dates
from models.utils.data_utils import get_measurement_variable
from models.utils.models_util import get_submodel_name, get_selected_submodel_name
from utils.timing import tic, toc
from utils.data_io import save_to_netcdf, load_data
from utils.logging import printf


#### ECMWF, Debiased ECMWF, PBC-ECMWF barplots by task (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

if False:
    print_improvements(metric_dic, 
                       model_name='pbc_ecmwf_combo', 
                       baseline_models=['ecmwf', 'debiased_ecmwf'])

fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   by_season=fig_by_season,
                   verbose=fig_verbose)

 #### ECMWF, Debiased ECMWF, PoET, Persistence++-PoET, Debias++-PoET, PBC-PoET barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'proj_perpp_msn', 'proj_tuned_msnpp', 'pbc_msn']
fig_model_names_str="PBC-PoET model components"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_msn_forecast"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
plot_rpss_barplot(metric_dic,
                model_names=fig_model_names,
                baseline_models=['ecmwf', 'debiased_ecmwf', 'msn'],
                target_dates=fig_target_dates,
                show_fig=fig_show,
                save_fig=fig_save,
                by_season=fig_by_season,
                verbose=fig_verbose,
                suffix='_breakdown')

 #### ECMWF, Debiased ECMWF, Persistence++-ECMWF, Debias++-ECMWF, PBC-ECMWF barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf',
                 'tuned_ecmwfpp', 'proj_tuned_ecmwfpp',
                 'perpp_ecmwf', 'proj_perpp_ecmwf',
                 'perpp_debias', 'proj_perpp_debias',
                 'pbc_ecmwf_combo']

fig_model_names_str="PBC-ECMWF model components"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

fig_variable_models = {
        "Temperature": [
            "ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_debias",
            "proj_perpp_debias",
            "pbc_ecmwf_combo",
        ],
        "Precipitation": [
            "ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_ecmwf",
            "proj_perpp_ecmwf",
            "pbc_ecmwf_combo",
        ],
        "Sea Level Pressure": [
            "ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_debias",
            "proj_perpp_debias",
            "pbc_ecmwf_combo",
        ],
    }
fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
fig_legend_order=["ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            # "perpp_debias",
            # "proj_perpp_debias",
            "perpp_ecmwf",
            "proj_perpp_ecmwf",
            "pbc_ecmwf_combo",
                 ]
plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   baseline_models=['ecmwf', 'debiased_ecmwf'],
                   variable_models=fig_variable_models,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   by_season=fig_by_season,
                   legend_order=fig_legend_order,
                   verbose=fig_verbose,
                   suffix='_breakdown')


#### ECMWF, Debiased ECMWF, PBC-ECMWF Diff maps (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_test"]
fig_verbose=False

# Set figure parameters
figure_model_names = ['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_metric = 'lat_lon_rpss'
figure_target_dates = 'std_test'
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
figure_show = True
figure_save = True

plot_metric_diff_grid_6x4(model_names=figure_model_names,
                        gt_ids=figure_gt_ids,
                        horizons=figure_horizons,
                        metric=figure_metric,
                        target_dates=figure_target_dates,
                        diff_cmap=diff_cmap,
                        skill_cmap=skill_cmap,   
                        show_fig=figure_show,
                        save_fig=figure_save)

#### AIFS vs. PBC-AIFS by task (2025)

In [ ]:
fig_model_names=['climatology', 'debiased_aifs', 'pbc_debias_aifs']
fig_model_names_str="AIFS models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_aifs_forecast"
fig_target_date_list=[fig_target_dates]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_date_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

if False:
    print_improvements(metric_dic, 
                       model_name='pbc_debias_aifs', 
                       baseline_models=['debiased_aifs'])

fig_show=True
fig_save=True

plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save) #, y_bottom=-0.2)

#### ECMWF, PoET, PBC-PoET barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'pbc_msn']
fig_model_names_str="MSN models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_msn_forecast"]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}


if False:
    print_improvements(metric_dic, 
                       model_name='pbc_msn', 
                       baseline_models=['msn', 'debiased_ecmwf'])

fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'pbc_msn']
fig_target_dates="std_msn_forecast"
fig_show=True
fig_save=True
fig_verbose=False
plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   verbose=fig_verbose)

#### Fuxi, PBC-ECMWF, barplots and diff maps (2017-2021)

In [ ]:
fig_model_names=['climatology', 'debiased_fuxi', 'pbc_ecmwf_combo']
fig_model_names_str="Fuxi models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_fuxi"]
common_dates=False
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      common_dates=common_dates,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

if False:
    print_improvements(metric_dic, 
                       model_name='pbc_ecmwf_combo', 
                       baseline_models=['debiased_fuxi'])

In [ ]:
fig_model_names=['climatology', 'debiased_fuxi', 'pbc_ecmwf_combo']
fig_target_dates="std_fuxi"
fig_show=True
fig_save=True
fig_verbose=False
plot_rpss_ci_barplot(metric_dic,
                     model_names=fig_model_names,
                     target_dates=fig_target_dates,
                     show_fig=fig_show,
                     save_fig=fig_save,
                     verbose=fig_verbose)

In [ ]:
# Set figure parameters
figure_model_names = ['debiased_fuxi', 'pbc_ecmwf_combo']
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_metric = 'lat_lon_rpss'
figure_target_dates = 'std_fuxi'
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
print_mean=False
figure_show = True
figure_save = True

plot_metric_diff_grid_6x3(model_names=figure_model_names,
                          gt_ids=figure_gt_ids,
                          horizons=figure_horizons,
                          metric=figure_metric,
                          target_dates=figure_target_dates,
                          diff_cmap=diff_cmap,
                          skill_cmap=skill_cmap, 
                          show_fig=figure_show,
                          save_fig=figure_save)

#### ECMWF, Deb. ECMWF, PBC-ECMWF RPSS barplots by region (std_test: 2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']###
fig_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
fig_horizons = [19, 26]
fig_target_dates = ["std_test"]
fig_regions = 'all' 
fig_verbose=False

metrics_dic = get_all_rps(
    model_names = fig_model_names,
    model_names_str="ECMWF-based models",
    horizons = fig_horizons,
    target_dates_list = fig_target_dates,
    regions = fig_regions,
    verbose=fig_verbose)
metrics_dic = {task: metrics_dic[(task,target_dates)] for task, target_dates in metrics_dic.keys()}


fig_model_names = ['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
fig_horizons = [19, 26]
fig_target_dates = "std_test"
fig_regions = 'all' 
fig_show = True
fig_save = True
fig_verbose = False

plot_rpss_by_region_all(metrics_dic,
                       model_names=fig_model_names,
                       gt_ids=fig_gt_ids,
                       horizons=fig_horizons,
                       target_dates=fig_target_dates,
                       regions=fig_regions,
                       show_fig=fig_show,
                       save_fig=fig_save,
                       verbose=fig_verbose)



#### ECMWF, Debiased-ECMWF, PBC-ECMWF RPSS by season barplots (2016-2024)

In [ ]:
fig_model_names=["climatology", "ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
fig_model_names_str="ECMWF-based models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_test"]
fig_verbose=False

metrics_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)
metrics_dic = {task: metrics_dic[(task,target_dates)] for task, target_dates in metrics_dic.keys()}

# Set figure parameters
figure_model_names = ["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_target_dates = 'std_test'
figure_show = True
figure_save = True
figure_verbose = False

plot_seasonal_rpss_grouped_bar(metrics_dic,
                                model_names=figure_model_names,
                                gt_ids=figure_gt_ids,
                                horizons=figure_horizons,
                                target_dates=figure_target_dates,
                                show_fig=figure_show,
                                save_fig=figure_save,
                                verbose=figure_verbose)

#### ECMWF, Debiased-ECMWF, PBC-ECMWF spatial bias (prediction-truth) (2016-2024)

In [ ]:
fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo", "gt"]
fig_gt_ids = ["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons = [19, 26]
fig_target_dates="std_test"
fig_vmin=-0.2
fig_vmax=0.2
fig_show_fig=True
fig_save_fig=True
fig_verbose=False

results_dict = get_all_preds(model_names=fig_model_names,
                  gt_ids=fig_gt_ids,
                  horizons=fig_horizons,
                  fs=[1, 2, 3, 4],
                  target_dates="std_test",
                  verbose=fig_verbose)

for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
    fig_show_cbar = (fig_horizon==26)
    plot_bias_maps_3x4(results_dict,  # Now taking the dictionary
                       model_names=fig_model_names,
                          gt_id=fig_gt_id,
                          horizon=fig_horizon,
                          fs=[1, 2, 3, 4],
                          cmap="RdBu_r",
                          vmin = fig_vmin,
                        vmax = fig_vmax,
                        show_cbar=fig_show_cbar,
                        show_fig = fig_show_fig,
                        save_fig = fig_save_fig
                    )

In [ ]:
if False:
    fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
    fig_gt_ids = ["era5-tas", "era5-pr", "era5-mslp"]
    fig_horizons = [19, 26]

    for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
        results = print_model_bias(results_dict, 
                        model_names=fig_model_names,
                        gt_id=fig_gt_id,
                        horizon=fig_horizon,
                        fs=[1, 2, 3, 4])